In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
import json
from functools import partial
import torch
from pathlib import Path
import pandas as pd
from torch.utils.data import DataLoader
from modules.tools import normalize, extract_model_name, Example, collate_batch, plot_metrics
from modules.train import train, evaluate
from modules.model import define_tokenizer, Detector, load_model
import gc


In [ ]:
def load_config(config_path):
    with open(config_path, "r") as f:
        return json.load(f)

In [ ]:
config = load_config(Path("configs/load_data.json"))

load_data_folder = Path(config["preprocess_folder"]) / "tokenized"
save_folder = Path(config["save_models_folder"])
dataset_name = config["dataset_for_experiment"]
features = config["features"]
models = config["models"]
batch_size = config["batch_size"]
epochs_num = config["epochs_num"]
learning_rate = config["learning_rate"]
device = config["device"]
fp16 = config["fp16"]

In [ ]:
def divide_data(dataset, features=None):
    if features is not None:
        _, preprocessor = normalize(dataset[dataset["split"] == "train"], features=features)
        dataset, preprocessor = normalize(dataset, features=features, preprocessor=preprocessor)
    else:
        preprocessor = None

    base_columns_X = ["input_ids", "attention_mask", "sample_id"]

    if features is None:
        columns_X = base_columns_X
    else:
        columns_X = base_columns_X + list(features)

    columns_y = "label"

    X_train = dataset[dataset["split"] == "train"][columns_X]
    X_val = dataset[dataset["split"] == "validate"][columns_X]
    X_test = dataset[dataset["split"] == "test"][columns_X]

    y_train = dataset[dataset["split"] == "train"][columns_y]
    y_val = dataset[dataset["split"] == "validate"][columns_y]
    y_test = dataset[dataset["split"] == "test"][columns_y]

    return X_train, X_val, X_test, y_train, y_val, y_test, preprocessor

In [ ]:


for model in models:
    model_name = extract_model_name(model)

    dataset_path = load_data_folder / f"{model_name}_{dataset_name}.parquet"
    dataset_df = pd.read_parquet(dataset_path)

    X_train_f, X_val_f, X_test_f, y_train_f, y_val_f, y_test_f, preprocessor = divide_data(dataset_df, features)
    X_train_n, X_val_n, X_test_n, y_train_n, y_val_n, y_test_n, _ = divide_data(dataset_df, features=None)

    tokenizer = define_tokenizer(model)
    collate_fn = partial(collate_batch, tokenizer=tokenizer)

    loader_train_f = DataLoader(Example(X_train_f, y_train_f, X_train_f[features]),
                                batch_size=batch_size,
                                shuffle=True,
                                collate_fn=collate_fn)
    loader_val_f = DataLoader(Example(X_val_f, y_val_f, X_val_f[features]),
                               batch_size=batch_size,
                               shuffle=False,
                               collate_fn=collate_fn)



    loader_train_n = DataLoader(Example(X_train_n, y_train_n, None),
                                batch_size=batch_size,
                                shuffle=True,
                                collate_fn=collate_fn)

    loader_val_n = DataLoader(Example(X_val_n, y_val_n, None),
                              batch_size=batch_size,
                              shuffle=False,
                              collate_fn=collate_fn)
    loader_test_f = DataLoader(Example(X_test_f, y_test_f, X_test_f[features]),
                                   batch_size=batch_size,
                                   shuffle=False,
                                   collate_fn=collate_fn)

    loader_test_n = DataLoader(Example(X_test_n, y_test_n, None),
                                   batch_size=batch_size,
                                   shuffle=False,
                                   collate_fn=collate_fn)


    model_f = Detector(model, tokenizer, max_token_length=512, device=device, preprocessor=preprocessor, feature_columns=features).to(device)


    optimizer_f = torch.optim.AdamW(model_f.parameters(), lr=learning_rate)

    criterion = torch.nn.CrossEntropyLoss()

    detector_f, results = train(epochs=epochs_num,
                                detector=model_f,
                                loader_train=loader_train_f,
                                loader_val=loader_val_f,
                                optimizer=optimizer_f,
                                criterion=criterion,
                                fp16=fp16,
                                priority="f1",
                                save_path=save_folder / f"{model_name}_{dataset_name}_f1_features",
                                eval_step=1000)

    results_f = evaluate(detector=detector_f, loader=loader_test_f, criterion=criterion)
    plot_metrics(results_f, x_column=None)
    with open(save_folder / f"{model_name}_{dataset_name}_f1_features" / "test_metrics.json", "w") as f:
        json.dump(results_f, f)

    del model_f, detector_f, optimizer_f
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


    model_n = Detector(model, tokenizer, max_token_length=512, device=device).to(device)
    optimizer_n = torch.optim.AdamW(model_n.parameters(), lr=learning_rate)



    detector_n, results = train(epochs=epochs_num,
                                detector=model_n,
                                loader_train=loader_train_n,
                                loader_val=loader_val_n,
                                optimizer=optimizer_n,
                                criterion=criterion,
                                fp16=fp16,
                                priority="f1",
                                save_path=save_folder / f"{model_name}_{dataset_name}_f1_textonly",
                                eval_step=1000)






    results_n = evaluate(detector=detector_n, loader=loader_test_n, criterion=criterion)
    plot_metrics(results_n, x_column=None)
    with open(save_folder / f"{model_name}_{dataset_name}_f1_textonly" / "test_metrics.json", "w") as f:
        json.dump(results_n, f)


    del model_n, detector_n, optimizer_n
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

